##### 코랩을 사용할 경우

In [1]:
try:
    # Google Drive를 Colab(/content/drive)rmfja 에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec06
    %ls
except ImportError:
    pass

##### 임포트

In [2]:
import torch
from torch.utils.data import Dataset
from sklearn.datasets import load_wine
import numpy as np

##### Device 설정

In [3]:
# Dataset은 기본적으로 CPU에서만 동작
# 사용하지는 않지만, Dataset은 CPU에서 동작한다는 것을 강조하기 위함
device = torch.device('cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### Dataset 클래스 작성

In [4]:
class WineDataset(Dataset):
    def __init__(self):
        """[필수] 데이터 로드 및 전처리 수행"""
        # 원본 데이터셋 로드
        wine = load_wine()
        
        # wine.data는 기본적으로 float64 (numpy의 기본 타입)
        # PyTorch 모델의 가중치 타입은 torch.float32 타입이므로, 입력 데이터도 같은 타입으로 변환
        # 변환 하지 않으면 torch.float64 → torch.float32로 변환 시 경고 발생
        data = wine.data.astype(np.float32)
        
        # 피처별 평균=0, 표준편차=1로 표준화
        # Wine 데이터셋은 피처 스케일이 크게 다름 (proline: 278~1680, malic_acid: 0.7~5.8)
        # 표준화하지 않으면 피처 스케일 차이로 인해 옵티마이저(경사하강법 기반)가 제대로 수렴하지 않아 학습이 안 됨
        # 어떤 특성의 표준편차가 0에 가까우면 값이 과도하게 커질 수 있음, 1e-8을 더해주므로서 완화시킴, 원래 값에 미치는 영향은 거의 없음
        mean = data.mean(axis=0)   # 피처별 평균, shape: (13,)
        std  = data.std(axis=0)    # 피처별 표준편차, shape: (13,)
        self.features = (data - mean) / (std + 1e-8)  # (178, 13)
        
        # 손실함수인 CrossEntropyLoss는 정수형(torch.long) 레이블을 요구하므로 int64로 변환
        # 텐서 변환시 np.int64 → torch.long으로 자동 변환됨
        self.labels = wine.target.astype(np.int64) # (178,)

    def __len__(self):
        """[필수] 전체 샘플 수 반환: len(dataset) 호출 시 실행되는 메소드"""
        return len(self.features)

    def __getitem__(self, idx):
        """[필수] 인덱스 idx에 해당하는 샘플 반환: dataset[idx] 호출 시 실행되는 메소드"""
        # ndarray → Tensor 변환: 모델 연산, 손실 함수, 역전파는 모두 Tensor 기반
        x = torch.tensor(self.features[idx])   # np.float32 → torch.float32
        y = torch.tensor(self.labels[idx])     # np.int64   → torch.long
        return x, y # 튜플 반환 (x: Tensor, y: Tensor)

##### Dataset 동작 확인

In [5]:
dataset = WineDataset()
print(f"전체 샘플 수: {len(dataset)}")  # __len__ 호출

x, y = dataset[0]  # __getitem__ 호출
print(f"첫 번째 샘플 피처: {repr(x)}")
print(f"첫 번째 샘플 레이블: {y}")

전체 샘플 수: 178
첫 번째 샘플 피처: tensor([ 1.5186, -0.5622,  0.2320, -1.1696,  1.9139,  0.8090,  1.0348, -0.6596,
         1.2249,  0.2517,  0.3622,  1.8479,  1.0130])
첫 번째 샘플 레이블: 0
